# CPDS-AI: YOLOv8 Adult vs Child Classification

Notebook này chạy trên Kaggle/Google Colab. Quy trình tải dataset từ Roboflow, fine-tune **YOLOv8n**, kiểm tra artifact và export ONNX. Không hard-code đường dẫn output của Ultralytics vì phiên bản mới có thể thay đổi cấu trúc thư mục.

In [1]:
!pip install -q ultralytics roboflow onnx
import ultralytics
ultralytics.checks()

Ultralytics 8.4.122 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7035.6/8062.4 GB disk)


## 1. Tải Dataset từ Roboflow

Trên Kaggle, tạo secret tên `ROBOFLOW_API_KEY` trong **Add-ons → Secrets**, bật quyền truy cập cho notebook. Không ghi API key trực tiếp vào notebook hoặc GitHub.

In [2]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
if not api_key:
    raise RuntimeError("Create Kaggle Secret ROBOFLOW_API_KEY and grant this notebook access.")

rf = Roboflow(api_key=api_key)
project = rf.workspace("timii-owolabi-pwfjm").project("child-adult-detection-bgjzk")
version = project.version(10)
dataset = version.download("yolov8")
dataset_path = Path(dataset.location)
data_yaml = dataset_path / "data.yaml"
if not data_yaml.is_file():
    raise FileNotFoundError(f"Roboflow export is missing data.yaml: {data_yaml}")

print("Đường dẫn dataset:", dataset_path)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to child-adult-detection-10 in yolov8:: 100%|██████████| 51780/51780 [00:06<00:00, 8363.76it/s] 


Đường dẫn dataset: /kaggle/working/child-adult-detection-10


## 2. Huấn luyện (Train) YOLOv8n

In [3]:
from ultralytics import YOLO

# Load pre-trained model nano
model = YOLO('yolov8n.pt')

# Kaggle GPU: device=0. Đổi sang device='cpu' nếu không bật Accelerator.
results = model.train(
    data=str(data_yaml), epochs=20, imgsz=640, device=0,
    project="/kaggle/working/cpds_runs", name="adult_child", exist_ok=True,
    patience=10, seed=42,
)

# Đây là nguồn duy nhất cho đường dẫn artifact. Không tự dựng chuỗi runs/...
run_dir = Path(results.save_dir)
best_weights = run_dir / "weights" / "best.pt"
if not best_weights.is_file():
    raise FileNotFoundError(f"Training did not create best.pt: {best_weights}")
print(f"Best weights: {best_weights}")

Ultralytics 8.4.122 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/child-adult-detection-10/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=adult_child

## 3. Export và smoke-test ONNX

Artifact cuối cùng luôn được copy về `/kaggle/working/artifacts/` để tải trong mục Output của Kaggle.

In [4]:
import json
import shutil
import onnx

# Recovery path: permits exporting an already-finished Kaggle run without retraining.
if "best_weights" not in globals():
    try:
        best_weights = Path(results.save_dir) / "weights" / "best.pt"
    except (NameError, AttributeError):
        candidates = sorted(Path("/kaggle/working").rglob("best.pt"), key=lambda path: path.stat().st_mtime)
        if not candidates:
            raise FileNotFoundError("No best.pt found. Run the training cell first.")
        best_weights = candidates[-1]
if not Path(best_weights).is_file():
    raise FileNotFoundError(f"Missing best weights: {best_weights}")

best_model = YOLO(str(best_weights))
# Fixed 640x640 shape is faster and more predictable on edge ONNX Runtime.
export_path = Path(best_model.export(format="onnx", imgsz=640, opset=12, dynamic=False, simplify=True))
onnx.checker.check_model(str(export_path))

artifacts_dir = Path("/kaggle/working/artifacts")
artifacts_dir.mkdir(exist_ok=True)
onnx_path = artifacts_dir / "best.onnx"
shutil.copy2(export_path, onnx_path)
metadata = {"class_names": best_model.names, "imgsz": 640, "source_weights": str(best_weights)}
(artifacts_dir / "vision_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

sample_images = list((dataset_path / "valid" / "images").glob("*"))
if sample_images:
    smoke_result = YOLO(str(onnx_path), task="detect")(str(sample_images[0]), verbose=False)[0]
    print(f"ONNX smoke test passed: {len(smoke_result.boxes)} detections on {sample_images[0].name}")
print(f"Download these Kaggle outputs: {onnx_path} and {artifacts_dir / 'vision_metadata.json'}")

Ultralytics 8.4.122 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/kaggle/working/cpds_runs/adult_child/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 305ms
 Downloaded onnxruntime
Prepared 2 packages in 311ms
Installed 2 packages in 13ms
 + onnxruntime==1.29.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: slimming with onnxslim 0.1.96...
ONNX